In [ ]:
PDF_PATH = "add your path here"

In [3]:
# Install dependencies
!pip install PyPDF2 pdfplumber pandas --quiet
print("✅ Ready")

✅ Ready



[notice] A new release of pip is available: 25.0.1 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import json, re, pandas as pd
from pathlib import Path
from dataclasses import dataclass, asdict, field
from typing import List, Dict, Any, Optional
from datetime import datetime
import PyPDF2, pdfplumber

@dataclass
class Section:
    id: str; number: Optional[str]; title: str; content: str
    level: int; parent_id: Optional[str]; page_number: Optional[int] = None
    subsections: List[str] = field(default_factory=list)
    references: List[str] = field(default_factory=list)
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class Table:
    id: str; title: str; location: str; headers: List[str]; rows: List[List[str]]
    page_number: Optional[int] = None; section_ref: Optional[str] = None
    metadata: Dict[str, Any] = field(default_factory=dict)

print("✅ Imports loaded")

✅ Imports loaded


In [5]:
class GenericDocProcessor:
    def __init__(self, pdf_path):
        self.pdf_path = Path(pdf_path)
        self.sections, self.tables, self.definitions = [], [], {}
        self.sec_counter = self.table_counter = 0
    
    def extract_text(self):
        text, pages = "", []
        try:
            with pdfplumber.open(self.pdf_path) as pdf:
                for i, page in enumerate(pdf.pages, 1):
                    t = page.extract_text() or ""
                    pages.append({'page': i, 'text': t})
                    text += t + "\n\n"
            print(f"✅ Extracted {len(pages)} pages")
        except: 
            with open(self.pdf_path, 'rb') as f:
                reader = PyPDF2.PdfReader(f)
                for i, page in enumerate(reader.pages, 1):
                    t = page.extract_text() or ""
                    pages.append({'page': i, 'text': t})
                    text += t + "\n\n"
            print(f"✅ Extracted {len(pages)} pages (PyPDF2)")
        
        self.pages = pages
        self.text = re.sub(r'\s+', ' ', text).strip()
        return self.text
    
    def extract_structure(self):
        lines = self.text.split('\n')
        current_chapter = current_part = None
        section_content = []; section_num = section_title = None
        
        for line in lines:
            line = line.strip()
            if not line: continue
            
            # Chapter
            if re.match(r'CHAPTER\s+[\dIVXLCDM]+', line, re.I):
                if section_content:
                    self._save_section(section_num, section_title, section_content, 3, current_part or current_chapter)
                    section_content = []
                current_chapter = line; current_part = None; continue
            
            # Part
            if re.match(r'PART\s+[\dIVXLCDM]+', line, re.I):
                if section_content:
                    self._save_section(section_num, section_title, section_content, 3, current_part or current_chapter)
                    section_content = []
                current_part = line; continue
            
            # Section number
            m = re.match(r'^(\d{1,3})\.\s+(.+)', line)
            if m:
                if section_content:
                    self._save_section(section_num, section_title, section_content, 3, current_part or current_chapter)
                section_num, section_title = m.groups()
                section_content = []
                continue
            
            if section_num: section_content.append(line)
        
        if section_content:
            self._save_section(section_num, section_title, section_content, 3, current_part or current_chapter)
        
        print(f"✅ Found {len(self.sections)} sections")
    
    def _save_section(self, num, title, content, level, parent):
        self.sec_counter += 1
        text = ' '.join(content)
        self.sections.append(Section(
            id=f"SEC_{self.sec_counter:04d}",
            number=num, title=title or "Untitled", content=text,
            level=level, parent_id=parent,
            metadata={'words': len(text.split()), 'chars': len(text)}
        ))
    
    def extract_tables(self):
        try:
            with pdfplumber.open(self.pdf_path) as pdf:
                for page_num, page in enumerate(pdf.pages, 1):
                    for table in page.extract_tables() or []:
                        if not table or len(table) < 2: continue
                        self.table_counter += 1
                        headers = [str(h or f"Col{i+1}").strip() for i, h in enumerate(table[0])]
                        rows = [[str(c or "").strip() for c in row] for row in table[1:]]
                        rows = [r for r in rows if any(r)]
                        if headers and rows:
                            self.tables.append(Table(
                                id=f"TABLE_{self.table_counter:04d}",
                                title=f"Table {self.table_counter}",
                                location=f"Page {page_num}",
                                headers=headers, rows=rows, page_number=page_num,
                                metadata={'num_rows': len(rows), 'num_cols': len(headers)}
                            ))
            print(f"✅ Found {len(self.tables)} tables")
        except Exception as e:
            print(f"⚠️ Table extraction: {e}")
    
    def extract_definitions(self):
        patterns = [r'"([^"]+)"\s+means\s+([^.;]+)', r'"([^"]+)"\s+includes\s+([^.;]+)']
        for p in patterns:
            for term, defn in re.findall(p, self.text, re.I):
                if term and defn: self.definitions[term.strip()] = defn.strip()
        print(f"✅ Found {len(self.definitions)} definitions")
    
    def process(self):
        print(f"\n🚀 Processing: {self.pdf_path.name}\n")
        self.extract_text()
        self.extract_structure()
        self.extract_tables()
        self.extract_definitions()
        
        return {
            'document_metadata': {
                'title': self.pdf_path.stem.replace('_', ' ').title(),
                'source': self.pdf_path.name,
                'pages': len(self.pages),
                'processed': datetime.now().isoformat()
            },
            'summary': {
                'total_sections': len(self.sections),
                'total_tables': len(self.tables),
                'total_definitions': len(self.definitions)
            },
            'sections': [asdict(s) for s in self.sections],
            'tables': [asdict(t) for t in self.tables],
            'definitions': self.definitions,
            'chunking_recommendations': {
                'chunk_size': '512-1024 tokens',
                'overlap': '50-100 tokens',
                'strategy': 'section-based'
            }
        }

print("✅ Processor ready")

✅ Processor ready


In [ ]:
# Process document
if Path(PDF_PATH).exists():
    processor = GenericDocProcessor(PDF_PATH)
    result = processor.process()
    print("\n✅ DONE\n")
else:
    print(f"File not found: {PDF_PATH}\n   Update PDF_PATH in first cell")


🚀 Processing: NIGERIA_TAX_ACT_2025.pdf

✅ Extracted 212 pages
✅ Found 0 sections
✅ Found 33 tables
✅ Found 227 definitions

✅ DONE



In [7]:
# Show summary
if 'result' in locals():
    s = result['summary']
    print(f"📊 Sections: {s['total_sections']}")
    print(f"📊 Tables: {s['total_tables']}")
    print(f"📊 Definitions: {s['total_definitions']}")
    print(f"\n📄 {result['document_metadata']['title']}")
    print(f"📃 {result['document_metadata']['pages']} pages")

📊 Sections: 0
📊 Tables: 33
📊 Definitions: 227

📄 Nigeria Tax Act 2025
📃 212 pages


In [8]:
# View sections
if 'result' in locals() and result['sections']:
    df = pd.DataFrame([{
        'Sec': s['number'] or '-',
        'Title': s['title'][:40],
        'Words': s['metadata']['words']
    } for s in result['sections'][:10]])
    print("\n📝 First 10 Sections:\n")
    print(df.to_string(index=False))
    if len(result['sections']) > 10:
        print(f"\n... +{len(result['sections'])-10} more")

In [9]:
# View tables
if 'result' in locals() and result['tables']:
    print("\n📊 Tables:\n")
    for i, t in enumerate(result['tables'][:2], 1):
        print(f"{i}. {t['title']} ({t['location']})")
        print(f"   {len(t['rows'])}×{len(t['headers'])}\n")
        print(pd.DataFrame(t['rows'][:3], columns=t['headers']).to_string(index=False))
        print()


📊 Tables:

1. Table 2 (Page 170)
   5×9

                    snoitpmexE ytreporp\n000,000,01N\not\nnaht\nsetaleR\nssel        Col3        Col4 ytreporp\n000,000,01N\not\nnaht\nsetaleR\nssel        Col6        Col7                    Col8        Col9
elbail\nytuD\nsnosreP\nyaP\not                                      eegagtroM  eerefsnarT  eerefsnarT                                      eegagtroM  eerefsnarT       eeyaP gnikat\nytiruces\nytraP     ynapmoC
                     etaR\nweN                                         %573.0        %5.1        %5.1                                         %573.0        %5.1       %01.0                  %573.0       %57.0
                          epyT                                    merolaV\ndA merolaV\ndA merolaV\ndA                                    merolaV\ndA merolaV\ndA merolaV\ndA             merolaV\ndA merolaV\ndA

2. Table 3 (Page 171)
   5×5

21 ot\nna tnemegnarra\ngnideecxe rof tnemesrubsid\na\neht ni\nrof deniatbo\nnosrep\nta desi

In [ ]:
# Save JSON
if 'result' in locals():
    output = Path("/mnt/user-data/outputs") / f"{Path(PDF_PATH).stem}_processed.json"
    output.parent.mkdir(exist_ok=True)
    
    with open(output, 'w') as f:
        json.dump(result, f, indent=2)
    
    size = output.stat().st_size / 1024
    print(f"\n💾 Saved: {output.name}")
    print(f"📊 Size: {size:.1f} KB")
    print(f"\n✅ Ready for chunking!")